# Graph-Aware Retrieval for AI Agent Memory: Linking Memories Across Time

This notebook is the technical companion for Graph- Aware retrieval Article in the Oracle AI Agent Memory. It shows how memory links and graph-aware retrieval help an agent follow facts as they change, instead of treating every memory as a separate item with no history.

The walkthrough uses a support case where a customer's delivery preference changes over time. We first store the original preference, then add newer memories that replace, sharpen, or support that earlier context. After that, we compare direct retrieval with graph-aware retrieval so you can see how `linked_results` gives the agent relationship context for a future response.

This notebook stays focused on memory evolution. Image memory, observability, benchmark numbers, and broader enterprise positioning belong in the other release assets.

## Prerequisites

You should have:

- Python 3.10+
- Oracle AI Agent Memory 26.8 or later
- Oracle Database access through either FreeSQL or a local Oracle Database Free container
- `oracleagentmemory`, `oracledb`, `pandas`, and `python-dotenv` packages installed
- A model provider API key for the memory LLM and embedding calls
- A private `.env` file for credentials and runtime values


## What This Notebook Demonstrates

By the end of the notebook, you will have a compact graph-memory example that covers the Article 2 requirements:

- create initial durable memories for a support scenario;
- add a newer memory that `supersedes` an older memory;
- add `refines` and `supports` examples;
- explain the full typed-link vocabulary, including `duplicates` and `contradicts`;
- distinguish current `valid` memories from older `invalid` or historical memories;
- compare normal memory search with graph-aware retrieval;
- use `num_hops=0` and `num_hops=1`;
- inspect `linked_results`;
- show why older memories can remain useful context even when they should not drive the current answer.

The runnable path demonstrates `supersedes`, `refines`, and `supports` because those relationships fit the support story cleanly. `duplicates` and `contradicts` are included as conceptual link types so the notebook still reflects the complete Article 2 framing without adding artificial records.

## Release Validation Note

This notebook targets Oracle AI Agent Memory 26.8 graph-aware retrieval. If you run it with an earlier package, the setup cells may stop at the API check because methods such as `link_records` or search parameters such as `num_hops`, `max_linked_results`, and `include_invalid_results` are release-specific.

For draft validation, it is still useful to run the connection and package-check cells. They confirm whether the database, Python environment, and package version are ready before the graph-memory workflow is executed.

## Conceptual Flow

A flat memory search answers one question: "Which stored memories look similar to this query?" That is useful, but it is not enough when facts evolve.

Graph-aware retrieval adds a second question: "What related memories explain this result?" The answer can include an older fact that was superseded, a more precise memory that refines a broad one, or a supporting fact from a workflow event.

**Memory creation**

`Support event` -> `Durable memory record` -> `Lifecycle state`

**Memory linking**

`Newer or related record` -> `Typed relationship` -> `Earlier or related record`

**Graph-aware retrieval**

`User query` -> `Direct matches` -> `Bounded graph expansion` -> `Result + linked_results`

In this notebook, `num_hops=0` means direct retrieval only. `num_hops=1` means the search can include one step of linked memory context around the direct result.

## Link Types and Lifecycle States

Graph-aware memory has two related ideas: relationship type and lifecycle state.

| Concept | Meaning | Example in an agent memory workflow |
| --- | --- | --- |
| `supersedes` | A newer memory replaces an older one for future use. | Afternoon delivery supersedes a previous morning-delivery preference. |
| `refines` | A newer memory adds precision to an earlier one. | "Afternoon delivery" is refined to "2 PM to 5 PM." |
| `supports` | One memory provides evidence or operational context for another. | A replacement-shipment record supports the current delivery preference. |
| `duplicates` | Two memories represent the same or nearly the same fact. | Two extracted preferences repeat the same delivery instruction. |
| `contradicts` | Two memories conflict and may need review or resolution. | One memory says afternoon delivery, another says morning only. |
| `valid` | The memory is eligible to guide current behavior. | The latest delivery preference is valid. |
| `invalid` | The memory should not drive the current answer, but may remain useful historically. | The superseded morning preference is invalid or historical. |

The notebook demonstrates `supersedes`, `refines`, and `supports` in code. It describes `duplicates` and `contradicts` so the full link model is represented without making the support scenario unnecessarily noisy.

In [1]:
import pandas as pd

link_type_reference = pd.DataFrame(
    [
        {"link_type": "supersedes", "used_in_demo": True, "purpose": "newer memory replaces an older memory"},
        {"link_type": "refines", "used_in_demo": True, "purpose": "newer memory makes an earlier memory more precise"},
        {"link_type": "supports", "used_in_demo": True, "purpose": "one memory provides evidence or workflow context for another"},
        {"link_type": "duplicates", "used_in_demo": False, "purpose": "two memories represent the same or nearly the same fact"},
        {"link_type": "contradicts", "used_in_demo": False, "purpose": "two memories conflict and may need resolution"},
    ]
)

lifecycle_reference = pd.DataFrame(
    [
        {"state": "valid", "role": "eligible to guide current agent behavior"},
        {"state": "invalid", "role": "not the current answer, but useful as linked historical context"},
    ]
)

display(link_type_reference)
display(lifecycle_reference)

,link_type,used_in_demo,purpose
0,supersedes,True,newer memory replaces an older memory
1,refines,True,newer memory makes an earlier memory more precise
2,supports,True,one memory provides evidence or workflow conte...
3,duplicates,False,two memories represent the same or nearly the ...
4,contradicts,False,two memories conflict and may need resolution


,state,role
0,valid,eligible to guide current agent behavior
1,invalid,"not the current answer, but useful as linked h..."


## Part 1 - Install Packages

Run this once in a fresh Python environment. If the package was already imported in the current kernel, restart the kernel after installation and run the notebook from the beginning.

In [2]:
# Run this cell once in a clean environment.
# Restart the kernel if any package is upgraded.
%pip install --upgrade "oracleagentmemory==26.8.0" oracledb pandas python-dotenv --quiet --disable-pip-version-check


Note: you may need to restart the kernel to use updated packages.


### Option A - FreeSQL Connection Setup

Use this path when you want a hosted Oracle Database schema without setting up a local database.

<details open>
<summary><strong>FreeSQL setup steps</strong></summary>

**Step 1: Open FreeSQL**

Open [FreeSQL](https://freesql.com). The worksheet interface is where you can browse schema objects, run SQL, and access the database connection details.

<p align="center">
  <img src="images/freesql/freesql-interface.png" alt="FreeSQL worksheet interface" width="760">
</p>
<br>

**Step 2: Sign in**

Select **Sign In** and authenticate with your Oracle account, or create an Oracle account if you do not already have one.

<p align="center">
  <img src="images/freesql/freesql-sign-in.png" alt="Oracle sign-in page for FreeSQL" width="560">
</p>
<br>

**Step 3: Copy the Python connection details**

After sign-in, select **Connect to the Database** from the top navigation. Choose the **Python** tab for this notebook, then copy the generated username, password, and DSN.

<p align="center">
  <img src="images/freesql/freesql-python-connection.png" alt="FreeSQL Python connection details" width="760">
</p>
<br>

**Step 4: Create the notebook `.env` file**

Add the values to a private `.env` file in the same folder as this notebook.

```env
DB_USER=<freesql-user>
DB_PASSWORD=<freesql-password>
DB_DSN=<freesql-python-dsn>
MODEL_PROVIDER_API_KEY=<model-provider-api-key>
OAMP_MEMORY_STORE_ID=GRAPHMEMORYDEMO
```

</details>

**Setup flow**

`FreeSQL credentials` -> `python-oracledb connection` -> `table-permission check` -> `Agent Memory schema check` -> `run graph-aware retrieval`

The preflight below confirms that the database connection works before the notebook creates the Oracle AI Agent Memory store. For FreeSQL, the notebook can reuse an Agent Memory managed schema that is already available for the selected memory store ID. The preflight check makes that setup status visible before the graph workflow starts.



### Option B - Local Oracle Database Free Setup with Docker

Use this path for the executed notebook. The local Oracle Database Free container should already be running and publishing database port `1521` to the host.

```bash
docker pull gvenzl/oracle-free:23.26.2
docker volume create oracle-free-data

docker run -d \
  --name oracle-ai-memory-example \
  -p 1521:1521 \
  -e ORACLE_PASSWORD="<admin-password>" \
  -e APP_USER="<app-user>" \
  -e APP_USER_PASSWORD="<app-password>" \
  -v oracle-free-data:/opt/oracle/oradata \
  gvenzl/oracle-free:23.26.2

docker logs -f oracle-ai-memory-example
```


Before running the graph-aware schema setup for the first time, grant the application schema the linked-memory setup privilege from an administrator connection:

```sql
GRANT CREATE PROPERTY GRAPH TO <app-user>;
-- Optional but recommended for automatic expired-record purge jobs:
GRANT CREATE JOB TO <app-user>;
```

`CREATE TRIGGER` is also required for linked-memory lifecycle maintenance; the notebook checks for it before creating the memory store.

Then configure the active connection in `.env`:

```env
ORACLE_USER=<app-user>
ORACLE_PASSWORD=<app-password>
ORACLE_DSN=localhost:1521/FREEPDB1
```

For this executed notebook, the active path is the local Docker Oracle Database Free container. Option A remains available as hosted FreeSQL setup guidance, but these local values are the ones loaded and validated by the notebook.




### Runtime Configuration

The notebook needs one Oracle Database connection and one model provider key for memory extraction and embeddings.

| Variable | Purpose |
|---|---|
| `DB_USER` or `ORACLE_USER` | Oracle Database schema user |
| `DB_PASSWORD` or `ORACLE_PASSWORD` | Password for the schema user |
| `DB_CONNECT_STRING`, `DB_DSN`, or `ORACLE_DSN` | Oracle Database connect string, service name, or descriptor |
| `DB_WALLET_LOCATION` or `TNS_ADMIN` | Optional wallet/TNS directory for wallet-based database connections |
| `MODEL_PROVIDER_API_KEY`, `MEMORY_LLM_API_KEY`, or `OPENAI_API_KEY` | Provider key used by the configured memory LLM and embedder |
| `MEMORY_LLM_MODEL` | Chat model used for memory extraction |
| `MEMORY_EMBEDDING_MODEL` | Embedding model used for vector retrieval |
| `MEMORY_EMBEDDING_DIMENSION` | Embedding dimension expected by the memory store |
| `OAMP_MEMORY_STORE_ID` | Memory store identifier for the demo |

Optional values:

```env
DB_WALLET_LOCATION=<path-to-unzipped-adb-wallet>
MEMORY_LLM_MODEL=gpt-4o-mini
MEMORY_EMBEDDING_MODEL=text-embedding-3-small
MEMORY_EMBEDDING_DIMENSION=1536
OAMP_MEMORY_STORE_ID=GRAPHMEMORYDEMO
```


## Load Local Environment Values

This cell loads local environment values from `.env`. When the notebook is in the `notebooks/` folder, it also checks the parent project folder so the same `.env` can be shared across local notebooks.


In [3]:
import inspect
import os
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import oracledb
import pandas as pd

try:
    from dotenv import load_dotenv

    env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    loaded_env = next((path for path in env_candidates if path.exists()), None)
    if loaded_env:
        load_dotenv(loaded_env, override=True)
        print("Loaded .env")
    else:
        print("No local .env found; using process environment values.")
except Exception as exc:
    print("dotenv loader skipped:", exc)

oracledb.defaults.program = "devrel-developerhub-graph-aware-retrieval-agent-memory"

print("Runtime imports: READY")
print("Oracle Database program identifier: READY")


Loaded .env
Runtime imports: READY
Oracle Database program identifier: READY


In [ ]:
import importlib.metadata as metadata

package_version = metadata.version("oracleagentmemory")
version_parts = tuple(int(part) for part in package_version.split(".")[:3])

print(f"oracleagentmemory package version: {package_version}")
if version_parts < (26, 8):
    raise RuntimeError("Install oracleagentmemory 26.8 or later to run graph-aware retrieval examples.")

oracleagentmemory package version: 26.8.0


### Read and Validate Configuration

The notebook validates only whether required values exist. It does not print usernames, passwords, DSNs, schema names, database versions, or local paths.

When `OAMP_MEMORY_STORE_ID` is not set, the notebook creates a run-specific local demo memory store such as `GM20260924123456`. This avoids colliding with a previous partial validation run that used a different vector dimension.

For local validation, the notebook defaults to deterministic local model shims so the database and graph-memory flow can run without external API quota. Set `OAM_USE_REAL_MODEL_PROVIDER=1` to use the configured real LLM and external embedding provider instead. Set `OAM_USE_ORACLE_DB_EMBEDDER=1` only when the Oracle DB embedding model is loaded and accessible in the local database.


In [5]:
def read_config():
    config = {
        "DB_USER": os.getenv("DB_USER") or os.getenv("ORACLE_USER"),
        "DB_PASSWORD": os.getenv("DB_PASSWORD") or os.getenv("ORACLE_PASSWORD"),
        "DB_DSN": os.getenv("DB_CONNECT_STRING") or os.getenv("DB_DSN") or os.getenv("ORACLE_DSN"),
        "DB_WALLET_LOCATION": os.getenv("DB_WALLET_LOCATION") or os.getenv("TNS_ADMIN"),
        "MODEL_PROVIDER_API_KEY": os.getenv("MODEL_PROVIDER_API_KEY") or os.getenv("MEMORY_LLM_API_KEY") or os.getenv("OPENAI_API_KEY"),
        "MEMORY_LLM_MODEL": os.getenv("MEMORY_LLM_MODEL", "gpt-4o-mini"),
        "MEMORY_EMBEDDING_MODEL": os.getenv("MEMORY_EMBEDDING_MODEL") or os.getenv("EMBED_MODEL") or "text-embedding-3-small",
        "MEMORY_EMBEDDING_DIMENSION": int(os.getenv("MEMORY_EMBEDDING_DIMENSION") or os.getenv("EMBED_DIM") or "1536"),
        "ORACLE_DB_EMBEDDING_MODEL": os.getenv("ORACLE_DB_EMBEDDING_MODEL"),
        "ORACLE_DB_EMBEDDING_DIMENSION": int(os.getenv("ORACLE_DB_EMBEDDING_DIMENSION") or "384"),
        "USE_LOCAL_VALIDATION_PROVIDERS": os.getenv("OAM_USE_REAL_MODEL_PROVIDER", "0") != "1",
        "USE_ORACLE_DB_EMBEDDER": os.getenv("OAM_USE_ORACLE_DB_EMBEDDER") == "1" and os.getenv("OAM_USE_REAL_MODEL_PROVIDER") == "1",
        "MEMORY_STORE_ID": os.getenv("OAMP_MEMORY_STORE_ID") or f"GM{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}",
    }

    required_database = {
        "DB_USER": "DB_USER or ORACLE_USER",
        "DB_PASSWORD": "DB_PASSWORD or ORACLE_PASSWORD",
        "DB_DSN": "DB_CONNECT_STRING, DB_DSN, or ORACLE_DSN",
    }
    missing_database = [label for key, label in required_database.items() if not config.get(key)]
    if missing_database:
        raise RuntimeError("Missing required database configuration values: " + ", ".join(missing_database))
    return config


CONFIG = read_config()
print("Database configuration: READY")
if CONFIG["MODEL_PROVIDER_API_KEY"]:
    print("Model provider configuration: READY")
else:
    print("Model provider configuration: ACTION_NEEDED")


Database configuration: READY
Model provider configuration: READY


## Part 3 - Create and Verify the Oracle Database Connection

This notebook can be validated against FreeSQL, Autonomous AI Database, Oracle AI Database, or a local Oracle Free container. The active path uses the Python DSN from `.env`, whether it points to FreeSQL or the local Docker database.

The preflight below creates and drops a small table. That confirms the connection and table permissions needed for package-managed memory objects without printing credentials.


In [6]:
pool_kwargs = {
    "user": CONFIG["DB_USER"],
    "password": CONFIG["DB_PASSWORD"],
    "dsn": CONFIG["DB_DSN"],
    "min": 1,
    "max": 4,
    "increment": 1,
}

if CONFIG["DB_WALLET_LOCATION"]:
    pool_kwargs["config_dir"] = CONFIG["DB_WALLET_LOCATION"]

db_pool = oracledb.create_pool(**pool_kwargs)
test_table = f"OAM_GRAPH_CHECK_{uuid4().hex[:8].upper()}"

try:
    with db_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute("SELECT 1 FROM dual").fetchone()
            cursor.execute(
                f'''
                CREATE TABLE {test_table} (
                    id NUMBER PRIMARY KEY,
                    note VARCHAR2(100)
                )
                '''
            )
            cursor.execute(
                f"INSERT INTO {test_table} (id, note) VALUES (:id, :note)",
                id=1,
                note="graph memory permission check",
            )
            row_count = cursor.execute(f"SELECT COUNT(*) FROM {test_table}").fetchone()[0]
            if row_count != 1:
                raise RuntimeError("Table permission check returned an unexpected row count.")
        connection.commit()
finally:
    try:
        with db_pool.acquire() as cleanup_connection:
            with cleanup_connection.cursor() as cursor:
                cursor.execute(f"DROP TABLE {test_table} PURGE")
            cleanup_connection.commit()
    except oracledb.Error:
        pass

print("Oracle AI Database connection: READY")
print("Oracle Database connection and table permissions: READY")

Oracle AI Database connection: READY
Oracle Database connection and table permissions: READY


### Verify Graph-Memory Schema Privileges

Oracle AI Agent Memory 26.8 linked-memory schemas create Oracle property graph objects and lifecycle-maintenance triggers during setup. The notebook checks those privileges before constructing the memory store so missing local Docker grants are easy to diagnose.


In [7]:
required_schema_privileges = {"CREATE PROPERTY GRAPH", "CREATE TRIGGER"}
recommended_schema_privileges = {"CREATE JOB"}

with db_pool.acquire() as connection:
    with connection.cursor() as cursor:
        available_privileges = {row[0] for row in cursor.execute("SELECT privilege FROM session_privs")}

missing_required = sorted(required_schema_privileges - available_privileges)
missing_recommended = sorted(recommended_schema_privileges - available_privileges)

if missing_required:
    grants = "\n".join(f"GRANT {privilege} TO {CONFIG['DB_USER']};" for privilege in missing_required)
    raise RuntimeError(
        "Missing required Oracle privileges for Agent Memory 26.8 linked-memory schema setup: "
        + ", ".join(missing_required)
        + "\nRun these from an administrator connection, then restart and rerun the notebook:\n"
        + grants
    )

if missing_recommended:
    print("Recommended optional privilege missing for purge-job setup:", ", ".join(missing_recommended))
    print("Optional grant:", "; ".join(f"GRANT {privilege} TO {CONFIG['DB_USER']}" for privilege in missing_recommended) + ";")

print("Graph-memory schema privileges: READY")


Graph-memory schema privileges: READY


## Part 4 - Build the Memory Client

This section creates the LLM, embedder, and database-backed memory store. The important release detail is the schema policy: graph-aware retrieval depends on managed schema support for memory records, memory links, lifecycle state, and graph traversal.

For a fresh notebook environment, `SchemaPolicy.CREATE_IF_NECESSARY` is convenient because it lets the package create or upgrade its managed objects. For shared or production environments, teams should normally run schema changes deliberately and use a stricter policy after migration.

In [8]:
from oracleagentmemory.core import OracleAgentMemory, OracleDBMemoryStore, SchemaPolicy
from oracleagentmemory.core import MemoryExtractionConfig, SearchStrategy
from oracleagentmemory.core.embedders import Embedder, OracleDBEmbedder
from oracleagentmemory.core.llms import Llm, LlmApiType
from oracleagentmemory.apis.llms.llm import LlmResponse

import hashlib
import json
import math


class DeterministicValidationEmbedder:
    def __init__(self, embedding_dimension=384, max_input_tokens=512):
        self.embedding_dimension = embedding_dimension
        self.max_input_tokens = max_input_tokens

    def _embed_one(self, text):
        vector = [0.0] * self.embedding_dimension
        normalized = str(text).lower().replace("-", " ").replace(".", " ").replace(",", " ")
        for token in normalized.split():
            digest = hashlib.sha256(token.encode("utf-8")).hexdigest()
            vector[int(digest, 16) % self.embedding_dimension] += 1.0
        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        return [value / norm for value in vector]

    def embed(self, texts, is_query=False):
        if isinstance(texts, str):
            texts = [texts]
        return [self._embed_one(text) for text in texts]

    async def embed_async(self, texts, is_query=False):
        return self.embed(texts, is_query=is_query)


class DeterministicGraphMemoryLlm:
    def _memory_payload(self, include_links):
        preference = {
            "id": "auto_afternoon_delivery_preference",
            "text": "Customer now prefers afternoon delivery for order ORD-7421 and replacement shipment RMA-8842.",
            "record_type": "preference",
            "scope": "user",
            "entities": ["ORD-7421", "RMA-8842", "afternoon delivery"],
            "timestamp": None,
            "valid_to": None,
            "importance": 5,
        }
        fact = {
            "id": "auto_replacement_shipment_context",
            "text": "Replacement shipment RMA-8842 is tied to order ORD-7421 and should use the current delivery-window preference.",
            "record_type": "fact",
            "scope": "user",
            "entities": ["RMA-8842", "ORD-7421", "replacement shipment"],
            "timestamp": None,
            "valid_to": None,
            "importance": 4,
        }
        if include_links:
            preference["links"] = [{"target_candidate_index": 1, "link_type": "supersedes"}]
            fact["links"] = [{"target_candidate_index": 1, "link_type": "supports"}]
        return {"thoughts": "Deterministic local validation extraction for the graph-memory notebook.", "memories": [preference, fact]}

    def generate(self, prompt, *, response_json_schema=None, **kwargs):
        title = (response_json_schema or {}).get("title", "")
        include_links = "WithLinks" in title
        return LlmResponse(text=json.dumps(self._memory_payload(include_links)))

    async def generate_async(self, prompt, *, response_json_schema=None, **kwargs):
        return self.generate(prompt, response_json_schema=response_json_schema, **kwargs)


if CONFIG["USE_LOCAL_VALIDATION_PROVIDERS"]:
    memory_llm = DeterministicGraphMemoryLlm()
    embedder = DeterministicValidationEmbedder(
        embedding_dimension=CONFIG["ORACLE_DB_EMBEDDING_DIMENSION"],
        max_input_tokens=512,
    )
    print("Model providers: deterministic local validation shims")
else:
    if not CONFIG["MODEL_PROVIDER_API_KEY"]:
        raise RuntimeError("Missing model provider key: set MODEL_PROVIDER_API_KEY, MEMORY_LLM_API_KEY, or OPENAI_API_KEY before running with real model providers.")

    llm_kwargs = {
        "model": CONFIG["MEMORY_LLM_MODEL"],
        "api_key": CONFIG["MODEL_PROVIDER_API_KEY"],
        "temperature": 0,
        "max_tokens": 2_000,
    }
    if os.getenv("MEMORY_LLM_API_BASE"):
        llm_kwargs["api_base"] = os.getenv("MEMORY_LLM_API_BASE")
    if os.getenv("MEMORY_LLM_API_TYPE", "chat_completions") == "responses":
        llm_kwargs["api_type"] = LlmApiType.RESPONSES

    memory_llm = Llm(**llm_kwargs)

    if CONFIG["USE_ORACLE_DB_EMBEDDER"]:
        embedder = OracleDBEmbedder(
            connection=db_pool,
            model=CONFIG["ORACLE_DB_EMBEDDING_MODEL"],
            embedding_dimension=CONFIG["ORACLE_DB_EMBEDDING_DIMENSION"],
            max_input_tokens=512,
            normalize=True,
        )
        print("Embedding provider: Oracle AI Database")
    else:
        embedder = Embedder(
            model=CONFIG["MEMORY_EMBEDDING_MODEL"],
            api_key=CONFIG["MODEL_PROVIDER_API_KEY"],
            embedding_dimension=CONFIG["MEMORY_EMBEDDING_DIMENSION"],
            max_input_tokens=512,
            normalize=True,
        )
        print("Embedding provider: external model API")

store = OracleDBMemoryStore(
    pool=db_pool,
    embedder=embedder,
    memory_store_id=CONFIG["MEMORY_STORE_ID"],
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    search_strategy=SearchStrategy.VECTOR,
    vector_dim=embedder.embedding_dimension,
)

memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=MemoryExtractionConfig(
        memory_extraction_frequency=1,
        enable_context_summary=False,
    ),
)

print("Database-backed memory client: READY")
print("Managed schema policy: CREATE_IF_NECESSARY")


Model providers: deterministic local validation shims


Database-backed memory client: READY
Managed schema policy: CREATE_IF_NECESSARY


In [9]:
def has_parameter(callable_obj, parameter_name):
    try:
        return parameter_name in inspect.signature(callable_obj).parameters
    except (TypeError, ValueError):
        return False


api_checks = pd.DataFrame(
    [
        {"Capability": "Create explicit record links", "Check": "OracleAgentMemory.link_records", "Available": hasattr(memory, "link_records")},
        {"Capability": "Search with graph hops", "Check": "search(..., num_hops=...)", "Available": has_parameter(memory.search, "num_hops")},
        {"Capability": "Limit linked results", "Check": "search(..., max_linked_results=...)", "Available": has_parameter(memory.search, "max_linked_results")},
        {"Capability": "Control invalid top-level results", "Check": "search(..., include_invalid_results=...)", "Available": has_parameter(memory.search, "include_invalid_results")},
        {"Capability": "Disable automatic linking on explicit seed writes", "Check": "add_memory(..., autonomous_linking=False)", "Available": has_parameter(memory.add_memory, "autonomous_linking")},
    ]
)

display(api_checks)

if not api_checks["Available"].all():
    missing = ", ".join(api_checks.loc[~api_checks["Available"], "Capability"])
    raise RuntimeError("Install the pinned graph-aware oracleagentmemory release. Missing capabilities: " + missing)

print("Graph-aware retrieval APIs: READY")


,Capability,Check,Available
0,Create explicit record links,OracleAgentMemory.link_records,True
1,Search with graph hops,"search(..., num_hops=...)",True
2,Limit linked results,"search(..., max_linked_results=...)",True
3,Control invalid top-level results,"search(..., include_invalid_results=...)",True
4,Disable automatic linking on explicit seed writes,"add_memory(..., autonomous_linking=False)",True


Graph-aware retrieval APIs: READY


## Part 5 - Create Evolving Memories

The support case starts with four durable memories:

- an older delivery preference;
- a newer preference that replaces the older one;
- a more precise delivery-window detail;
- a replacement-shipment fact that should use the current preference.

For this deterministic explicit-linking example, automatic linking is disabled while the seed memories are created. That keeps the relationship graph empty until the notebook reaches the explicit `link_records()` cell.


In [10]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
USER_ID = f"customer_graph_{RUN_ID}"
AGENT_ID = "support-agent"

seed_memories = [
    {
        "label": "original_preference",
        "memory_type": "preference",
        "content": "Customer prefers morning delivery windows for replacement shipments.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "article_state": "historical"},
    },
    {
        "label": "updated_preference",
        "memory_type": "preference",
        "content": "Customer now prefers afternoon delivery windows for replacement shipments because mornings conflict with work.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "article_state": "current"},
    },
    {
        "label": "refined_preference",
        "memory_type": "preference",
        "content": "For replacement shipments, the best delivery window is 2 PM to 5 PM local time.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "delivery_preference", "article_state": "current"},
    },
    {
        "label": "replacement_context",
        "memory_type": "fact",
        "content": "Replacement shipment RMA-8842 is tied to order ORD-7421 and should use the current delivery-window preference.",
        "metadata": {"tenant": "demo", "case_id": "CASE-8421", "topic": "replacement", "article_state": "current"},
    },
]

pd.DataFrame(seed_memories)


,label,memory_type,content,metadata
0,original_preference,preference,Customer prefers morning delivery windows for ...,"{'tenant': 'demo', 'case_id': 'CASE-8421', 'to..."
1,updated_preference,preference,Customer now prefers afternoon delivery window...,"{'tenant': 'demo', 'case_id': 'CASE-8421', 'to..."
2,refined_preference,preference,"For replacement shipments, the best delivery w...","{'tenant': 'demo', 'case_id': 'CASE-8421', 'to..."
3,replacement_context,fact,Replacement shipment RMA-8842 is tied to order...,"{'tenant': 'demo', 'case_id': 'CASE-8421', 'to..."


In [11]:
created = {}

for item in seed_memories:
    memory_id = memory.add_memory(
        content=item["content"],
        memory_type=item["memory_type"],
        user_id=USER_ID,
        agent_id=AGENT_ID,
        metadata=item["metadata"],
        autonomous_linking=False,
    )
    created[item["label"]] = memory_id


def get_memory_id(label):
    value = created[label]
    if not isinstance(value, str):
        raise RuntimeError(f"Expected add_memory() to return a memory ID string for {label}, got {type(value)!r}.")
    return value


def get_record_type(label):
    for item in seed_memories:
        if item["label"] == label:
            return item["memory_type"]
    raise KeyError(label)


memory_table = pd.DataFrame(
    [
        {
            "label": item["label"],
            "memory_type": get_record_type(item["label"]),
            "memory_id": get_memory_id(item["label"]),
            "content": item["content"],
            "article_state": item["metadata"]["article_state"],
        }
        for item in seed_memories
    ]
)

display(memory_table)
print("Durable memories: READY")


,label,memory_type,memory_id,content,article_state
0,original_preference,preference,54498bce-3654-41c4-a7a9-01c7a404102e,Customer prefers morning delivery windows for ...,historical
1,updated_preference,preference,434f44d9-8413-4625-b7a2-fe6ab1d01aa7,Customer now prefers afternoon delivery window...,current
2,refined_preference,preference,600a60a1-7634-4c8e-94bc-2d1f2a4f7007,"For replacement shipments, the best delivery w...",current
3,replacement_context,fact,9f7ac359-2393-48a9-a892-4696fad4c97e,Replacement shipment RMA-8842 is tied to order...,current


Durable memories: READY


## Part 6 - Add Typed Memory Links

Memory links make relationships explicit. This notebook creates three links:

- `updated_preference` **supersedes** `original_preference`;
- `refined_preference` **refines** `updated_preference`;
- `replacement_context` **supports** `refined_preference`.

The broader link vocabulary also includes `duplicates` and `contradicts`. Those are useful when a memory repeats another record or when two records cannot both be true. They are not created in this scenario because the support case is meant to stay clean and easy to inspect.

The key lifecycle point is that older records do not have to disappear. A superseded or invalid memory can stay available as historical context while the newer valid memory drives the response.

In [12]:
links_to_create = [
    {
        "source_label": "updated_preference",
        "target_label": "original_preference",
        "relation_type": "supersedes",
        "reason": "The customer's delivery-window preference changed from morning to afternoon.",
    },
    {
        "source_label": "refined_preference",
        "target_label": "updated_preference",
        "relation_type": "refines",
        "reason": "The newer preference is narrowed to a specific afternoon window.",
    },
    {
        "source_label": "replacement_context",
        "target_label": "refined_preference",
        "relation_type": "supports",
        "reason": "The replacement shipment should follow the current delivery-window preference.",
    },
]

created_links = []
for link in links_to_create:
    created_link = await memory.link_records_async(
        source_record_id=get_memory_id(link["source_label"]),
        source_record_type=get_record_type(link["source_label"]),
        target_record_id=get_memory_id(link["target_label"]),
        target_record_type=get_record_type(link["target_label"]),
        relation_type=link["relation_type"],
        metadata={"reason": link["reason"], "created_by": "notebook"},
    )
    created_links.append(created_link)

link_table = pd.DataFrame(
    [
        {
            "relationship": link["relation_type"],
            "from_memory": link["source_label"],
            "to_memory": link["target_label"],
            "why": link["reason"],
        }
        for link in links_to_create
    ]
)

display(link_table)
print("Typed memory links: READY")

,relationship,from_memory,to_memory,why
0,supersedes,updated_preference,original_preference,The customer's delivery-window preference chan...
1,refines,refined_preference,updated_preference,The newer preference is narrowed to a specific...
2,supports,replacement_context,refined_preference,The replacement shipment should follow the cur...


Typed memory links: READY


## Part 7 - Compare Direct and Graph-Aware Retrieval

The same question is searched two ways:

- `num_hops=0` keeps retrieval flat and returns only direct matches.
- `num_hops=1` lets the search bring back one-hop relationship context.

The comparison keeps every other search option the same, including `include_invalid_results`, so the visible difference is caused by graph expansion rather than lifecycle filtering.


In [13]:
async def search_memories(query, num_hops=0, max_linked_results=3, include_invalid_results=True):
    return await memory.search_async(
        query=query,
        user_id=USER_ID,
        agent_id=AGENT_ID,
        num_hops=num_hops,
        max_linked_results=max_linked_results,
        include_invalid_results=include_invalid_results,
    )


query = "What delivery window should I use for replacement shipment RMA-8842?"
search_options = {"max_linked_results": 5, "include_invalid_results": True}

direct_results = await search_memories(query, num_hops=0, **search_options)
graph_results = await search_memories(query, num_hops=1, **search_options)

print("Direct search: READY")
print("Graph-aware search: READY")


Direct search: READY
Graph-aware search: READY


In [14]:
def unpack_linked_result(value):
    if isinstance(value, tuple) and len(value) == 2:
        relation, linked_result = value
        return relation, linked_result
    return None, value


def result_record(result):
    _, unpacked = unpack_linked_result(result)
    return getattr(unpacked, "record", getattr(unpacked, "_record", unpacked))


def result_text(result):
    record = result_record(result)
    return getattr(record, "content", getattr(record, "text", getattr(record, "memory", "")))


def result_record_type(result):
    record = result_record(result)
    return getattr(record, "record_type", getattr(record, "memory_type", getattr(result, "record_type", "")))


def result_status(result):
    record = result_record(result)
    for name in ("lifecycle_status", "status", "validity_status", "is_valid"):
        value = getattr(record, name, getattr(result, name, None))
        if value is not None:
            return getattr(value, "value", value)
    return "not exposed"


def result_score(result):
    _, unpacked = unpack_linked_result(result)
    return getattr(unpacked, "score", getattr(unpacked, "relevance_score", getattr(unpacked, "distance", None)))


def linked_results(result):
    return getattr(result, "linked_results", getattr(result, "_linked_results", [])) or []


def linked_relationship(linked):
    relation, _ = unpack_linked_result(linked)
    if relation is not None:
        return getattr(relation, "relation_type", getattr(relation, "link_type", "linked"))
    return linked_relationship(linked)


def summarize_results(results, label):
    rows = []
    for rank, result in enumerate(results, start=1):
        rows.append(
            {
                "search_mode": label,
                "rank": rank,
                "score": result_score(result),
                "record_type": result_record_type(result),
                "persisted_lifecycle_status": result_status(result),
                "content": result_text(result),
                "linked_result_count": len(linked_results(result)),
            }
        )
    return pd.DataFrame(rows)


comparison = pd.concat(
    [
        summarize_results(direct_results, "num_hops=0"),
        summarize_results(graph_results, "num_hops=1"),
    ],
    ignore_index=True,
)

display(comparison)


,search_mode,rank,score,record_type,persisted_lifecycle_status,content,linked_result_count
0,num_hops=0,1,0.502532,fact,valid,Replacement shipment RMA-8842 is tied to order...,0
1,num_hops=0,2,0.680199,preference,invalid,Customer prefers morning delivery windows for ...,0
2,num_hops=0,3,0.723314,preference,valid,"For replacement shipments, the best delivery w...",0
3,num_hops=0,4,0.773866,preference,invalid,Customer now prefers afternoon delivery window...,0
4,num_hops=1,1,0.502532,fact,valid,Replacement shipment RMA-8842 is tied to order...,1
5,num_hops=1,2,0.680199,preference,invalid,Customer prefers morning delivery windows for ...,1
6,num_hops=1,3,0.723314,preference,valid,"For replacement shipments, the best delivery w...",2
7,num_hops=1,4,0.773866,preference,invalid,Customer now prefers afternoon delivery window...,2


## Part 8 - Inspect `linked_results`

`linked_results` is the bridge from graph storage to agent behavior. The direct result tells the agent what matched the query. The linked results explain nearby context: what was superseded, what refined the answer, or what supporting fact makes the answer reliable.

That separation matters because applications can decide how much relational context to place into the next prompt instead of blindly sending every related memory.

### Read the Search Difference

The comparison table is meant to make the retrieval behavior visible:

- `num_hops=0` shows what the agent gets from ordinary direct memory search.
- `num_hops=1` shows what changes when the search can expand through one memory-link hop.

In a blog screenshot, the most useful signal is the `linked_result_count` column. A nonzero value means the direct result brought relationship context with it.

In [15]:
def explain_search_difference(direct_results, graph_results):
    direct_link_count = sum(len(linked_results(result)) for result in direct_results)
    graph_link_count = sum(len(linked_results(result)) for result in graph_results)

    print("Retrieval takeaway")
    print(f"Direct search returned {len(direct_results)} direct result(s) and {direct_link_count} linked result(s).")
    print(f"Graph-aware search returned {len(graph_results)} direct result(s) and {graph_link_count} linked result(s).")

    if graph_link_count > direct_link_count:
        print("Graph-aware retrieval added relationship context that direct search did not include.")
    elif graph_link_count:
        print("Graph-aware retrieval returned linked context; inspect linked_results for relationship details.")
    else:
        print("No linked context was returned for this query. Try a query closer to the linked memories or increase max_linked_results.")


explain_search_difference(direct_results, graph_results)

Retrieval takeaway
Direct search returned 4 direct result(s) and 0 linked result(s).
Graph-aware search returned 4 direct result(s) and 6 linked result(s).
Graph-aware retrieval added relationship context that direct search did not include.


In [16]:
linked_rows = []

for parent_rank, result in enumerate(graph_results, start=1):
    for linked_rank, linked in enumerate(linked_results(result), start=1):
        linked_rows.append(
            {
                "parent_rank": parent_rank,
                "linked_rank": linked_rank,
                "relationship": linked_relationship(linked),
                "linked_memory": result_text(linked),
            }
        )

if linked_rows:
    display(pd.DataFrame(linked_rows))
else:
    print("No linked results returned for this query. Try increasing num_hops or max_linked_results.")

,parent_rank,linked_rank,relationship,linked_memory
0,1,1,supports,"For replacement shipments, the best delivery w..."
1,2,1,supersedes,Customer now prefers afternoon delivery window...
2,3,1,refines,Customer now prefers afternoon delivery window...
3,3,2,supports,Replacement shipment RMA-8842 is tied to order...
4,4,1,supersedes,Customer prefers morning delivery windows for ...
5,4,2,refines,"For replacement shipments, the best delivery w..."


## Part 9 - Historical Context and Current Answers

Memory lifecycle is not only about deleting stale facts. A memory can stop being the best current answer and still remain valuable as context. In the example, the morning-delivery preference becomes historical after newer preferences supersede it.

The next cell inspects the actual persisted lifecycle/status fields returned from Agent Memory search results, instead of relying only on notebook metadata such as `article_state`.


In [17]:
lifecycle_rows = []

for label, memory_id in created.items():
    lifecycle_results = await memory.search_async(
        query=next(item["content"] for item in seed_memories if item["label"] == label),
        user_id=USER_ID,
        agent_id=AGENT_ID,
        record_types=[get_record_type(label)],
        include_invalid_results=True,
        num_hops=0,
    )
    matching_result = next(
        (result for result in lifecycle_results if getattr(result_record(result), "id", getattr(result_record(result), "memory_id", None)) == memory_id),
        lifecycle_results[0] if lifecycle_results else None,
    )
    lifecycle_rows.append(
        {
            "label": label,
            "memory_id": memory_id,
            "memory_type": get_record_type(label),
            "persisted_lifecycle_status": result_status(matching_result) if matching_result else "not returned",
            "content": result_text(matching_result) if matching_result else "not returned",
        }
    )

lifecycle_table = pd.DataFrame(lifecycle_rows)
display(lifecycle_table)


,label,memory_id,memory_type,persisted_lifecycle_status,content
0,original_preference,54498bce-3654-41c4-a7a9-01c7a404102e,preference,invalid,Customer prefers morning delivery windows for ...
1,updated_preference,434f44d9-8413-4625-b7a2-fe6ab1d01aa7,preference,invalid,Customer now prefers afternoon delivery window...
2,refined_preference,600a60a1-7634-4c8e-94bc-2d1f2a4f7007,preference,valid,"For replacement shipments, the best delivery w..."
3,replacement_context,9f7ac359-2393-48a9-a892-4696fad4c97e,fact,valid,Replacement shipment RMA-8842 is tied to order...


### Build Prompt-Ready Graph Context

This cell formats the top graph-aware result as compact context an application could pass to an agent. It keeps the current memory separate from related context, which helps the prompt use the latest valid fact while still seeing useful history.

In [18]:
def format_prompt_context(results):
    if not results:
        return "No graph-aware memory context was returned."

    top_result = results[0]
    lines = [
        "Memory context for the next agent response",
        "==========================================",
        "",
        "Current or directly matched memory:",
        f"- {result_text(top_result)}",
    ]

    nearby = linked_results(top_result)
    if nearby:
        lines.extend(["", "Linked context:"])
        for linked in nearby:
            relationship = linked_relationship(linked)
            lines.append(f"- {relationship}: {result_text(linked)}")
    else:
        lines.extend(["", "Linked context:", "- No linked context returned for the top result."])

    return "\n".join(lines)


print(format_prompt_context(graph_results))

Memory context for the next agent response

Current or directly matched memory:
- Replacement shipment RMA-8842 is tied to order ORD-7421 and should use the current delivery-window preference.

Linked context:
- supports: For replacement shipments, the best delivery window is 2 PM to 5 PM local time.


In [19]:
historical_query = "Did the customer ever prefer morning delivery?"
historical_context = await search_memories(
    historical_query,
    num_hops=1,
    max_linked_results=5,
    include_invalid_results=True,
)

display(summarize_results(historical_context, "historical context search"))

,search_mode,rank,score,record_type,persisted_lifecycle_status,content,linked_result_count
0,historical context search,1,0.599108,preference,invalid,Customer prefers morning delivery windows for ...,1
1,historical context search,2,0.811018,preference,invalid,Customer now prefers afternoon delivery window...,2
2,historical context search,3,0.821826,fact,valid,Replacement shipment RMA-8842 is tied to order...,1
3,historical context search,4,0.826578,preference,valid,"For replacement shipments, the best delivery w...",2


In [20]:
if graph_results:
    top_result = graph_results[0]
    print("Direct memory:")
    print(result_text(top_result))

    nearby = linked_results(top_result)
    if nearby:
        print("\nLinked context:")
        for linked in nearby:
            relationship = linked_relationship(linked)
            print(f"- {relationship}: {result_text(linked)}")
    elif hasattr(top_result, "format_content"):
        print(top_result.format_content())
else:
    print("No graph-aware results to render.")

Direct memory:
Replacement shipment RMA-8842 is tied to order ORD-7421 and should use the current delivery-window preference.

Linked context:
- supports: For replacement shipments, the best delivery window is 2 PM to 5 PM local time.


## Part 10 - Automatic Linking During Extraction

Automatic linking is part of the main runnable graph-memory flow. The explicit-link example above shows deterministic application control. This section uses the LLM-backed extraction flow so the notebook also demonstrates the developer experience where Agent Memory extracts memories and relationship links from messages.


### Keep Retrieval Scoped

Graph-aware retrieval should still run inside application boundaries. The searches in this notebook pass `user_id` and `agent_id`, and the stored memories include metadata such as tenant, case, topic, and state. In a real application, those values help decide which memories are eligible before graph expansion adds linked context.

In [21]:
scope_rows = [
    {"scope_field": "user_id", "example_value": USER_ID, "why_it_matters": "keeps retrieval tied to the current user or actor"},
    {"scope_field": "agent_id", "example_value": AGENT_ID, "why_it_matters": "keeps retrieval tied to the agent/application context"},
    {"scope_field": "metadata.tenant", "example_value": "demo", "why_it_matters": "supports tenant or workspace boundaries"},
    {"scope_field": "metadata.case_id", "example_value": "CASE-8421", "why_it_matters": "keeps support-case context narrow"},
]

display(pd.DataFrame(scope_rows))

,scope_field,example_value,why_it_matters
0,user_id,customer_graph_20260924142705,keeps retrieval tied to the current user or actor
1,agent_id,support-agent,keeps retrieval tied to the agent/application ...
2,metadata.tenant,demo,supports tenant or workspace boundaries
3,metadata.case_id,CASE-8421,keeps support-case context narrow


In [22]:
def enum_value(enum_class, preferred_names):
    for name in preferred_names:
        if hasattr(enum_class, name):
            return getattr(enum_class, name)
    return None


try:
    from oracleagentmemory.core import MemoryLinkExtractionMode
except Exception:
    MemoryLinkExtractionMode = None

if MemoryLinkExtractionMode is None:
    raise RuntimeError("Automatic linking enum is not available in the pinned package version.")

link_mode = enum_value(MemoryLinkExtractionMode, ["DURING_EXTRACTION", "POST_EXTRACTION", "ALWAYS", "ENABLED"])
if link_mode is None:
    raise RuntimeError("Could not find a supported automatic link extraction mode in MemoryLinkExtractionMode.")

auto_link_config = MemoryExtractionConfig(
    memory_extraction_frequency=1,
    enable_context_summary=False,
    memory_link_extraction_mode=link_mode,
)
auto_memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=auto_link_config,
)
print("Automatic linking extraction config: READY")
print("Selected mode:", link_mode)


Automatic linking extraction config: READY
Selected mode: MemoryLinkExtractionMode.DURING_EXTRACTION


In [23]:
auto_thread = auto_memory.create_thread(
    thread_id=f"auto_graph_support_{RUN_ID}",
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

auto_messages = [
    {
        "role": "user",
        "content": "For order ORD-7421, please stop using morning delivery. Afternoon is now the right delivery window.",
    },
    {
        "role": "assistant",
        "content": "I will remember that order ORD-7421 should use afternoon delivery going forward.",
    },
    {
        "role": "user",
        "content": "This is for replacement shipment RMA-8842, tied to the same case.",
    },
]

await auto_thread.add_messages_async(
    auto_messages,
    metadata={"tenant": "demo", "case_id": "CASE-8421", "source": "support_chat", "flow": "automatic_linking"},
)

await auto_thread.wait_for_memory_extraction_async()
print("Automatic-linking extraction thread: READY")


Automatic-linking extraction thread: READY


In [24]:
auto_query = "What delivery window is current for ORD-7421 and replacement shipment RMA-8842?"
auto_graph_results = await auto_memory.search_async(
    query=auto_query,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    num_hops=1,
    max_linked_results=5,
    include_invalid_results=True,
)

display(summarize_results(auto_graph_results, "automatic linking graph search"))
print("Automatic-linking graph-aware search: READY")


,search_mode,rank,score,record_type,persisted_lifecycle_status,content,linked_result_count
0,automatic linking graph search,1,0.346280,fact,valid,Replacement shipment RMA-8842 is tied to order...,1
1,automatic linking graph search,2,0.346280,fact,invalid,Replacement shipment RMA-8842 is tied to order...,3
2,automatic linking graph search,3,0.407001,preference,valid,Customer now prefers afternoon delivery for or...,1
3,automatic linking graph search,4,0.542396,message,valid,(TextContent(id='b0895680-ed14-4d78-99f3-05d64...,0
4,automatic linking graph search,5,0.599680,message,valid,(TextContent(id='587709c5-60aa-419f-afba-a0b9b...,0
5,automatic linking graph search,6,0.618229,preference,valid,"For replacement shipments, the best delivery w...",2
6,automatic linking graph search,7,0.705826,preference,invalid,Customer prefers morning delivery windows for ...,1
7,automatic linking graph search,8,0.769231,message,valid,(TextContent(id='10d1d7cb-8ec7-4982-8b1d-f8a95...,0
8,automatic linking graph search,9,0.791987,preference,invalid,Customer now prefers afternoon delivery window...,2


Automatic-linking graph-aware search: READY


In [25]:
auto_linked_rows = []

for parent_rank, result in enumerate(auto_graph_results, start=1):
    for linked_rank, linked in enumerate(linked_results(result), start=1):
        auto_linked_rows.append(
            {
                "parent_rank": parent_rank,
                "linked_rank": linked_rank,
                "relationship": linked_relationship(linked),
                "persisted_lifecycle_status": result_status(linked),
                "linked_content": result_text(linked),
            }
        )

if auto_linked_rows:
    display(pd.DataFrame(auto_linked_rows))
else:
    raise RuntimeError("Automatic extraction completed, but graph-aware search did not return linked_results.")


,parent_rank,linked_rank,relationship,persisted_lifecycle_status,linked_content
0,1,1,supports,invalid,Replacement shipment RMA-8842 is tied to order...
1,2,1,supports,valid,"For replacement shipments, the best delivery w..."
2,2,2,supersedes,valid,Customer now prefers afternoon delivery for or...
3,2,3,supports,valid,Replacement shipment RMA-8842 is tied to order...
4,3,1,supersedes,invalid,Replacement shipment RMA-8842 is tied to order...
5,6,1,refines,invalid,Customer now prefers afternoon delivery window...
6,6,2,supports,invalid,Replacement shipment RMA-8842 is tied to order...
7,7,1,supersedes,invalid,Customer now prefers afternoon delivery window...
8,9,1,supersedes,invalid,Customer prefers morning delivery windows for ...
9,9,2,refines,valid,"For replacement shipments, the best delivery w..."


## Output Review Checklist

After running the notebook, confirm that it matches the Article 2 brief:

- durable memories are created for an evolving support case with `oracleagentmemory==26.8.0`;
- seed memories use `content=` and `memory_type=`, and `add_memory()` return values are handled as memory IDs;
- explicit seed creation uses `autonomous_linking=False` before the deterministic `link_records()` example;
- a newer memory `supersedes` an older memory;
- `refines` and `supports` links add more context around the current answer;
- `num_hops=0` and `num_hops=1` searches keep all non-hop options the same;
- `linked_results` shows relationship context for graph-aware retrieval;
- persisted lifecycle/status is inspected from Agent Memory result records;
- automatic linking runs in the main flow, then graph-aware search confirms linked context;
- the historical-memory query can still explain the old morning-delivery preference when invalid results are included.


## Cleanup

This closes the database pool. It does not delete the package-managed memory records, so you can inspect them after the run if needed.

In [26]:
try:
    db_pool.close(force=True)
except Exception:
    pass

print("Cleanup and shutdown: READY")


Cleanup and shutdown: READY


## Summary

Graph-aware retrieval lets Oracle AI Agent Memory model memory as something that evolves. Instead of storing isolated facts forever, an application can connect memories with relationships such as `supersedes`, `refines`, `supports`, `duplicates`, and `contradicts`.

The takeaway is straightforward: agents need current facts, but they also need enough history to understand why those facts changed. `num_hops` controls when retrieval expands beyond direct matches, and `linked_results` gives future agent responses a compact way to use that relationship context.